<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>Cómo funcionan las redes neuronales</b></font></h1>
<h2><b>Notebook 3: </b>Transfer learning</h2>

In [ ]:
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
from torchvision import datasets, models, transforms

import utils

torch.manual_seed(42)

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("trabajaremos en:", DISPOSITIVO)

----

<br>

## **Parte 0:** Nuestro objetivo

En *Silicon Valley* —la serie— Jian-Yang promete **SeeFood**, "el Shazam de la comida": usted le toma una foto a un plato y la app le dice qué es. Cuando por fin la demuestran, resulta que la app solo sabe responder dos cosas: **hot dog** o **no hot dog**.

Eso es justo lo que vamos a construir: un modelo que mire una foto y diga si eso es un perro caliente o no.

Con un problema. En el cuaderno anterior quedó claro que una red con más capacidad que datos no aprende, **memoriza**. Y las imágenes son el caso extremo: una foto de 224x224 son 150.528 números de entrada, y nosotros vamos a tener unos cientos de ejemplos. Las redes que entienden imágenes de verdad se entrenan con **millones** de fotos, semanas de GPU y una cuenta de luz que no queremos pagar.

La salida es no empezar de cero: agarrar una red que alguien más ya entrenó con esos millones de fotos y **reciclarle el conocimiento**. Eso es **transfer learning**, y es como se resuelven hoy casi todos los problemas de visión.

----

<br>

## **Parte 1:** Los datos

Usaremos el dataset de **hot dogs**, armado con fotos de ImageNet: 1.400 fotos de perros calientes y 1.400 de comida que no lo es (que es la parte difícil: sánduches, tacos, salchichas sueltas).

Viene repartido en carpetas, y el nombre de la carpeta **es** la etiqueta:

```
data/hotdog/
├── train/
│   ├── hotdog/        1.000 fotos
│   └── not-hotdog/    1.000 fotos
└── test/
    ├── hotdog/          400 fotos
    └── not-hotdog/      400 fotos
```

Esa es la forma estándar de guardar imágenes en PyTorch: **una carpeta por clase**. `ImageFolder` la lee sola, sin que haya que explicarle nada.

> La descarga son **261 MB** y se hace una sola vez: queda guardada en `data/hotdog/`.

In [ ]:
URL = "http://d2l-data.s3-accelerate.amazonaws.com/hotdog.zip"
CARPETA = Path("data/hotdog")

if not (CARPETA / "train").exists():
    comprimido = CARPETA.parent / "hotdog.zip"

    if not comprimido.exists():
        print("descargando 261 MB (solo la primera vez)...")
        urllib.request.urlretrieve(URL, comprimido)

    print("descomprimiendo...")
    with zipfile.ZipFile(comprimido) as archivo:
        archivo.extractall(CARPETA.parent)
    comprimido.unlink()

if not (CARPETA / "train").exists():
    raise FileNotFoundError(f"El zip no dejó {CARPETA / 'train'}; mire cómo quedó {CARPETA.parent}")

for parte in ("train", "test"):
    for clase in sorted(p for p in (CARPETA / parte).iterdir() if p.is_dir()):
        print(f"{parte}/{clase.name}: {sum(1 for _ in clase.iterdir())} fotos")

### Cómo se le da una foto a una red

Una red no recibe un `.jpg`: recibe un tensor. Y no cualquiera, sino uno con **exactamente la forma con la que se entrenó** la red que vamos a reciclar:

| Paso | Qué hace |
|---|---|
| `Resize(256)` | lleva el lado corto de la foto a 256 píxeles |
| `CenterCrop(224)` | recorta el centro: todas quedan de 224x224 |
| `ToTensor()` | de foto a tensor `(3, 224, 224)` con valores entre $0$ y $1$ |
| `Normalize(media, desv)` | le resta la media y la divide por la desviación de **ImageNet** |

Ese último paso parece un detalle y no lo es: densenet aprendió a mirar fotos centradas en esos números. Si le cambiamos la escala, le estamos mostrando un mundo distinto del que conoce.

Del dataset completo vamos a tomar solo una parte —**200 fotos de cada clase** para entrenar y 100 para validar— por dos razones: es lo que pasa en la vida real (nadie tiene millones de fotos etiquetadas de *su* problema), y es lo que hace que este cuaderno corra en un computador sin GPU.

In [ ]:
FOTOS_POR_CLASE = 200     # para entrenar, de cada clase
FOTOS_DE_PRUEBA = 100     # para validar, de cada clase

NOMBRES = ("no hot dog", "hot dog")

MEDIA = [0.485, 0.456, 0.406]   # las de ImageNet: con estas se entrenó densenet
DESV = [0.229, 0.224, 0.225]

# Lo que espera densenet.
transformacion = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, DESV),
])

# Una versión más barata, para la red que entrenaremos desde cero.
transformacion_chica = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])

In [ ]:
def indices_balanceados(carpeta, por_clase, semilla=42):
    """Las mismas `por_clase` fotos de cada clase: al azar, pero siempre las mismas."""
    etiquetas = torch.tensor(carpeta.targets)
    generador = torch.Generator().manual_seed(semilla)

    elegidas = []
    for clase in range(len(carpeta.classes)):
        indices = (etiquetas == clase).nonzero().flatten()
        orden = torch.randperm(len(indices), generator=generador)[:por_clase]
        elegidas.append(indices[orden])

    return torch.cat(elegidas).tolist()


class SoloHotDog(Dataset):
    """Las carpetas, recortadas y con la etiqueta puesta como la queremos: 1 = hot dog."""

    def __init__(self, ruta, transformacion, por_clase):
        carpeta = datasets.ImageFolder(ruta, transform=transformacion)

        self.clases = carpeta.classes
        self.positiva = [i for i, c in enumerate(self.clases) if "not" not in c.lower()][0]
        self.fotos = Subset(carpeta, indices_balanceados(carpeta, por_clase))

    def __len__(self):
        return len(self.fotos)

    def __getitem__(self, i):
        foto, clase = self.fotos[i]
        return foto, torch.tensor([float(clase == self.positiva)])


entrenamiento = SoloHotDog(CARPETA / "train", transformacion, FOTOS_POR_CLASE)
validacion = SoloHotDog(CARPETA / "test", transformacion, FOTOS_DE_PRUEBA)

# Las mismas fotos, en pequeño: la semilla es la misma, así que la selección también.
entrenamiento_chico = SoloHotDog(CARPETA / "train", transformacion_chica, FOTOS_POR_CLASE)
validacion_chica = SoloHotDog(CARPETA / "test", transformacion_chica, FOTOS_DE_PRUEBA)

print("carpetas:", entrenamiento.clases, "-> la clase positiva es",
      entrenamiento.clases[entrenamiento.positiva])
print(f"{len(entrenamiento)} fotos para entrenar, {len(validacion)} para validar")

In [ ]:
fotos, etiquetas = next(iter(DataLoader(validacion, batch_size=12, shuffle=True)))

utils.dibujar_muestra(
    fotos,
    [NOMBRES[int(e)] for e in etiquetas],
    titulo="Doce fotos de validación, ya recortadas a 224x224",
)

----

<br>

## **Parte 2:** ¿Con quién vamos a comparar?

Como en el cuaderno del Titanic, antes de traer la artillería hay que tener contra qué medirla. Acá la línea base es **hacerlo por las malas**: una red convolucional pequeña, entrenada desde cero con nuestras 400 fotos.

Una **convolución** es un filtro que se barre por toda la imagen buscando un patrón —un borde, una mancha, una textura— y devuelve un mapa de dónde lo encontró. Se apilan varias, con un `MaxPool` entre ellas que va reduciendo la imagen a la mitad, y al final lo que queda es un puñado de números que resumen la foto. Ese resumen entra a las capas de siempre.

Para que no se demore una eternidad le damos las fotos en **96x96** en vez de 224x224. Le estamos poniendo el problema más fácil y más barato que a la otra: aun así, va a perder.

In [ ]:
class CNNDesdeCero(nn.Module):
    """Cuatro bloques de convolución y un clasificador. Todo por aprender."""

    def __init__(self):
        super().__init__()

        def bloque(entra, sale):
            return nn.Sequential(
                nn.Conv2d(entra, sale, kernel_size=3, padding=1),   # barre la foto con filtros
                nn.ReLU(),                                          # la que dobla
                nn.MaxPool2d(2),                                    # y la reduce a la mitad
            )

        self.cuerpo = nn.Sequential(
            bloque(3, 16),      # 96 -> 48
            bloque(16, 32),     # 48 -> 24
            bloque(32, 64),     # 24 -> 12
            bloque(64, 64),     # 12 -> 6
        )
        self.clasificador = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 6 * 6, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),     # un solo número de salida: el logit de "es hot dog"
        )

    def forward(self, x):
        return self.clasificador(self.cuerpo(x))

### Las métricas, otra vez

Son las mismas cuatro del cuaderno anterior —accuracy, precision, recall y f1— con la matriz de confusión de la que salen todas. Lo único que cambia es el significado del positivo: acá **positivo = es un hot dog**.

In [ ]:
def metricas(y_real, y_pred):
    vp = torch.sum((y_pred == 1) & (y_real == 1))
    fp = torch.sum((y_pred == 1) & (y_real == 0))
    fn = torch.sum((y_pred == 0) & (y_real == 1))
    vn = torch.sum((y_pred == 0) & (y_real == 0))

    precision = vp / (vp + fp)
    recall = vp / (vp + fn)

    return {
        "accuracy": ((vp + vn) / y_real.numel()).item(),
        "precision": precision.item(),
        "recall": recall.item(),
        "f1": (2 * precision * recall / (precision + recall)).item(),
        "matriz": torch.tensor([[vn, fp], [fn, vp]]),
    }


@torch.no_grad()
def predecir(modelo, cargador):
    """Pasa un conjunto entero por el modelo: devuelve las etiquetas y las probabilidades."""
    modelo.eval()
    reales, probabilidades = [], []

    for x, y in cargador:
        probabilidades.append(torch.sigmoid(modelo(x.to(DISPOSITIVO))).cpu())
        reales.append(y)

    return torch.cat(reales), torch.cat(probabilidades)


def evaluar(modelo, cargador, umbral=0.5):
    y_real, probabilidades = predecir(modelo, cargador)
    return metricas(y_real, (probabilidades >= umbral).float())

In [ ]:
@torch.no_grad()
def perdida_en(modelo, cargador, funcion_perdida):
    """La pérdida promedio sobre un conjunto, sin tocar los parámetros."""
    modelo.eval()
    acumulada = 0.0

    for x, y in cargador:
        acumulada += funcion_perdida(modelo(x.to(DISPOSITIVO)), y.to(DISPOSITIVO)).item() * len(x)

    return acumulada / len(cargador.dataset)


def entrenar(modelo, datos_entreno, datos_prueba, epocas, tasa=1e-3,
             parametros=None, mostrar_cada=1):
    """El bucle de siempre, anotando la pérdida en los dos conjuntos."""
    modelo.to(DISPOSITIVO)
    funcion_perdida = nn.BCEWithLogitsLoss()
    optimizador = optim.Adam(
        modelo.parameters() if parametros is None else parametros, lr=tasa)

    historial = {"entrenamiento": [], "validación": []}

    for epoca in range(epocas):
        modelo.train()
        acumulada = 0.0

        for x, y in datos_entreno:
            x, y = x.to(DISPOSITIVO), y.to(DISPOSITIVO)
            optimizador.zero_grad()                       # 1. borrar los gradientes viejos
            perdida = funcion_perdida(modelo(x), y)       # 2-3. predecir y medir el error
            perdida.backward()                            # 4. calcular gradientes
            optimizador.step()                            # 5. actualizar parámetros
            acumulada += perdida.item() * len(x)

        historial["entrenamiento"].append(acumulada / len(datos_entreno.dataset))
        historial["validación"].append(perdida_en(modelo, datos_prueba, funcion_perdida))

        if (epoca + 1) % mostrar_cada == 0:
            print(f"época {epoca + 1:>3} — entrenamiento: {historial['entrenamiento'][-1]:.4f}"
                  f"   validación: {historial['validación'][-1]:.4f}")

    return historial

In [ ]:
torch.manual_seed(42)

cnn = CNNDesdeCero()
print(f"{sum(p.numel() for p in cnn.parameters()):,} parámetros, todos por aprender\n")

historial_cero = entrenar(
    cnn,
    DataLoader(entrenamiento_chico, batch_size=32, shuffle=True),
    DataLoader(validacion_chica, batch_size=32),
    epocas=15,
)

utils.dibujar_metricas(
    historial_cero,
    titulo="Desde cero, época a época",
    y_etiqueta="pérdida (BCE)",
)

resultados_cero = evaluar(cnn, DataLoader(validacion_chica, batch_size=32))

utils.dibujar_resultados(resultados_cero, titulo="Desde cero — conjunto de validación",
                         nombres=NOMBRES)

### Lo que acaba de pasar

La curva de **entrenamiento** baja: la red se está aprendiendo las 400 fotos. La de **validación** baja un poco y se queda pegada, o se devuelve para arriba — el mismo cruce del cuaderno anterior, ahora con fotos.

Y la accuracy queda rondando el **60-70%**. Tirar una moneda da 50%. Es decir: con 400 fotos, una red que arranca sabiendo nada de nada apenas le gana a la suerte.

No está mal programada. Es que no hay con qué: los bordes, las texturas y las formas que hacen falta para *ver* no se aprenden con 400 ejemplos.

----

<br>

## **Parte 3:** Reciclar el trabajo de otro

**DenseNet-161** es una red convolucional que ya está entrenada. La entrenaron con **ImageNet**: 1,2 millones de fotos repartidas en 1.000 categorías, en un entrenamiento que ningún portátil aguanta.

Lo valioso no es que sepa las 1.000 categorías. Es que **para poder saberlas tuvo que aprender a ver**:

- Las primeras capas detectan bordes, esquinas y cambios de color.
- Las del medio, texturas y pedazos de cosas: rejillas, pelo, ruedas, pan.
- Las últimas, objetos completos.

Nada de eso es exclusivo de ImageNet. Un borde es un borde en cualquier foto del mundo, y la textura del pan de un perro caliente se parece muchísimo a la del pan de una hamburguesa. Ese trabajo ya está hecho y es gratis.

Transfer learning es exactamente esto: **quedarse con el cuerpo de la red y cambiarle la última capa** por una que responda nuestra pregunta.

In [ ]:
pesos = models.DenseNet161_Weights.IMAGENET1K_V1   # ~110 MB, se descargan la primera vez
densenet = models.densenet161(weights=pesos)

print(densenet.classifier, "<- la capa que decide entre las 1.000 clases de ImageNet\n")
print(f"{sum(p.numel() for p in densenet.parameters()):,} parámetros, todos ya entrenados")

Antes de tocarle nada, preguntémosle qué ve en nuestras fotos. Responde con las categorías de ImageNet, y entre esas mil hay una que se llama *hotdog*.

In [ ]:
categorias = pesos.meta["categories"]

densenet.to(DISPOSITIVO).eval()
with torch.no_grad():
    salidas = densenet(fotos[:6].to(DISPOSITIVO)).cpu()

utils.dibujar_muestra(
    fotos[:6],
    [categorias[i] for i in salidas.argmax(dim=1)],
    columnas=6,
    titulo="Lo que dice densenet antes de que la toquemos",
)

----

<br>

## **Parte 4:** El cuerpo como extractor de características

Dos operaciones y ya está listo el reciclaje:

1. **Congelar el cuerpo** — `requires_grad = False` en cada parámetro. Así el optimizador no los toca: lo que la red aprendió de ImageNet se queda intacto.
2. **Quitarle la cabeza** — cambiamos `classifier` por `nn.Identity()`, que no hace nada. En vez de 1.000 puntajes, la red pasa a devolver los **2.208 números** con los que resume la foto justo antes de decidir.

Esos 2.208 números son las **características** de la foto: "hay pan", "hay algo alargado y rojizo", "hay textura de pasto". Ya no son 150.528 píxeles, son 2.208 conceptos.

Y acá va el truco que hace que esto corra sin GPU: **como el cuerpo está congelado, esos 2.208 números nunca cambian**. No hay que recalcularlos en cada época. Se calculan **una sola vez** y se guardan.

> La pasada completa son unas 600 fotos a ~0,3 s cada una: **unos 3 minutos** en CPU. Después de esto, cada época de entrenamiento demora menos de un segundo.

In [ ]:
for parametro in densenet.parameters():
    parametro.requires_grad = False       # 1. el cuerpo no se vuelve a tocar

densenet.classifier = nn.Identity()       # 2. sin cabeza: la salida son las características
densenet.to(DISPOSITIVO).eval()


@torch.no_grad()
def extraer(datos):
    """Pasa las fotos por el cuerpo congelado, una sola vez, y guarda los vectores."""
    caracteristicas, etiquetas = [], []

    for x, y in DataLoader(datos, batch_size=32):
        caracteristicas.append(densenet(x.to(DISPOSITIVO)).cpu())
        etiquetas.append(y)

    return torch.cat(caracteristicas), torch.cat(etiquetas)


inicio = time.time()

X_entreno, y_entreno = extraer(entrenamiento)
X_val, y_val = extraer(validacion)

print(f"listo en {time.time() - inicio:.0f} segundos")
print("cada foto quedó convertida en un vector de", X_entreno.shape[1], "números")
print("entrenamiento:", tuple(X_entreno.shape), "| validación:", tuple(X_val.shape))

In [ ]:
caracteristicas_entreno = DataLoader(TensorDataset(X_entreno, y_entreno),
                                     batch_size=32, shuffle=True)
caracteristicas_val = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

----

<br>

## **Parte 5:** La cabeza nueva

Lo único que falta es una red que, a partir de esos 2.208 números, responda una sola cosa: ¿hot dog o no?

Y esa red es diminuta — de hecho es la misma del cuaderno 1, un par de capas lineales con una ReLU en la mitad. Ahí está la gracia del asunto: **el problema difícil (ver) ya lo resolvió otro; el nuestro (decidir) es fácil.**

In [ ]:
torch.manual_seed(42)

cabeza = nn.Sequential(
    nn.Linear(2208, 256),     # de las 2.208 características a 256
    nn.ReLU(),
    nn.Dropout(0.3),          # apaga neuronas al azar para que no se memorice las fotos
    nn.Linear(256, 1),        # un solo número de salida: el logit de "es hot dog"
)

print(f"cuerpo congelado:  {sum(p.numel() for p in densenet.parameters()):>12,}  (ninguno se entrena)")
print(f"cabeza nueva:      {sum(p.numel() for p in cabeza.parameters()):>12,}  (estos sí)")
print(f"la CNN desde cero: {sum(p.numel() for p in cnn.parameters()):>12,}")

In [ ]:
historial_transfer = entrenar(
    cabeza,
    caracteristicas_entreno,
    caracteristicas_val,
    epocas=40,
    mostrar_cada=5,
)

utils.dibujar_metricas(
    historial_transfer,
    titulo="La cabeza nueva, época a época",
    y_etiqueta="pérdida (BCE)",
)

resultados_transfer = evaluar(cabeza, caracteristicas_val)

utils.dibujar_resultados(resultados_transfer, titulo="Transfer learning — conjunto de validación",
                         nombres=NOMBRES)

----

<br>

## **Parte 6:** Cara a cara

Las mismas fotos, la misma partición, las mismas métricas. Lo único distinto es de dónde arrancó cada una.

In [ ]:
utils.dibujar_comparacion(
    {
        "desde cero": resultados_cero,
        "transfer learning": resultados_transfer,
    },
    titulo="Conjunto de validación",
)

----

<br>

## **Parte 7:** SeeFood

Ya está la app. Una foto entra, pasa por el cuerpo prestado, la cabeza nueva decide.

In [ ]:
fotos_prueba, etiquetas_prueba = next(iter(DataLoader(validacion, batch_size=12, shuffle=True)))

cabeza.eval()
with torch.no_grad():
    probabilidades = torch.sigmoid(cabeza(densenet(fotos_prueba.to(DISPOSITIVO)))).cpu()

utils.dibujar_predicciones(
    fotos_prueba,
    etiquetas_prueba,
    probabilidades,
    nombres=NOMBRES,
    titulo="SeeFood en acción (verde = acertó)",
)

In [ ]:
@torch.no_grad()
def see_food(ruta):
    """La app completa: una foto, un veredicto."""
    densenet.eval()
    cabeza.eval()

    foto = Image.open(ruta).convert("RGB")
    entrada = transformacion(foto).unsqueeze(0).to(DISPOSITIVO)
    probabilidad = torch.sigmoid(cabeza(densenet(entrada))).item()

    utils.dibujar_veredicto(foto.resize((224, 224)), probabilidad, nombres=NOMBRES)
    return probabilidad


# Cambie la ruta por la de cualquier foto suya y vuelva a correr la celda.
carpeta_hotdog = CARPETA / "test" / entrenamiento.clases[entrenamiento.positiva]
foto_de_prueba = sorted(carpeta_hotdog.iterdir())[7]

see_food(foto_de_prueba)

----

<br>

## **Cierre**

Con 400 fotos, entrenar desde cero deja la accuracy rondando el **60-70%**. Reciclando a densenet la cosa se va por encima del **90%**, entrenando menos de un millón de parámetros y sin tocar los 26 millones del cuerpo.

Y no es que nuestra cabeza sea más inteligente que la CNN desde cero: es que **partió de un modelo que ya sabía ver**. Las 1,2 millones de fotos de ImageNet no fueron nuestras, pero el conocimiento que dejaron sí.

Eso deja tres ideas para llevarse:

- **Casi nadie entrena desde cero.** Se parte de un modelo preentrenado y se le adapta el final. En visión, en audio y en texto.
- **Congelar es la versión barata.** El siguiente paso, cuando hay más datos, es el *fine-tuning*: descongelar las últimas capas del cuerpo y seguir entrenando con una tasa de aprendizaje bien pequeña, para ajustar lo que la red ya sabe sin borrarlo.
- **Es lo mismo que pasa con los modelos de lenguaje.** Nadie entrena un GPT desde cero para responder correos: se toma un modelo base, carísimo de entrenar, y se le ajusta el final con unos pocos ejemplos propios.

<br>

> **La moraleja:** en el cuaderno 2 la red perdió porque el problema no daba para tanta capacidad. Acá el problema sí daba, pero los datos no. Cuando los datos son pocos y el problema es duro, la salida no es una red más grande — es empezar desde el trabajo de alguien más.

-----

<br>

### **Siguiente notebook:**

[`04_tictactoe.ipynb`](04_tictactoe.ipynb)

### <font color="#29c4d9">**Notebook 3 listo.**</font>

<br>

---

<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>